In [3]:
!pip install pandas numpy scikit-learn joblib

In [4]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors

from google.colab import files

In [5]:
uploaded = files.upload()

Saving xgboost_model_v2.pkl to xgboost_model_v2.pkl
Saving clean_data.csv to clean_data.csv


In [6]:
df = pd.read_csv("clean_data.csv")

In [7]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp").reset_index(drop=True)

encoder = LabelEncoder()

df["risk_level"] = encoder.fit_transform(df["risk_level"])

for col in ["operator_action","transition_stage"]:

    if col in df.columns:

        df[col] = LabelEncoder().fit_transform(df[col])

In [8]:
lag_columns = [
    "stock_flow",
    "steam_pressure",
    "machine_speed",
    "moisture",
    "actual_basis_weight"
]

for col in lag_columns:

    df[f"{col}_lag1"] = df[col].shift(1)

    df[f"{col}_lag2"] = df[col].shift(2)

    df[f"{col}_lag3"] = df[col].shift(3)

df["bw_mean_5"] = df["actual_basis_weight"].rolling(5).mean()

df["steam_mean_5"] = df["steam_pressure"].rolling(5).mean()

df["speed_mean_5"] = df["machine_speed"].rolling(5).mean()

FUTURE_STEPS=60

df["future_basis_weight"]=df["actual_basis_weight"].shift(-FUTURE_STEPS)

df=df.dropna().reset_index(drop=True)

In [9]:
model=joblib.load("xgboost_model_v2.pkl")

In [10]:
X=df.drop(columns=[
    "timestamp",
    "actual_basis_weight",
    "future_basis_weight"
])

In [11]:
knn=NearestNeighbors(
    n_neighbors=5,
    metric="euclidean"
)

knn.fit(X)

NearestNeighbors(metric='euclidean')

In [12]:
sample=X.iloc[[800]]

In [13]:
distance,index=knn.kneighbors(sample)

index

array([[ 800,   89, 3041,  119, 2332]])

In [14]:
similar=df.iloc[index[0]]

similar

,timestamp,recipe_id,target_basis_weight,actual_basis_weight,stock_flow,filler_flow,steam_pressure,machine_speed,moisture,ash,...,moisture_lag1,moisture_lag2,moisture_lag3,actual_basis_weight_lag1,actual_basis_weight_lag2,actual_basis_weight_lag3,bw_mean_5,steam_mean_5,speed_mean_5,future_basis_weight
800,2026-01-01 09:07:00,103,80,79.85,95.86,16.84,120.51,954.79,6.07,14.35,...,5.83,5.16,6.03,80.54,81.47,80.25,80.664,120.226,965.682,78.78
89,2026-01-01 08:07:45,103,80,79.22,95.64,17.85,119.83,954.95,6.07,14.40,...,4.97,5.77,5.75,80.13,80.64,78.69,79.746,120.034,970.070,79.44
3041,2026-01-01 12:13:45,103,80,80.57,102.25,12.18,119.67,957.06,5.01,12.34,...,4.91,5.16,5.56,79.73,78.42,79.33,79.878,119.620,966.268,78.91
119,2026-01-01 08:10:15,103,80,79.18,96.24,14.24,122.65,957.39,5.66,14.70,...,5.31,5.43,5.86,80.08,81.31,79.69,79.966,121.612,972.722,78.58
2332,2026-01-01 11:14:40,103,80,79.63,95.79,14.04,121.32,954.35,5.34,8.94,...,5.12,6.12,4.96,80.77,81.33,79.62,80.152,119.676,971.544,80.88


In [15]:
recommendation=[]

row=sample.iloc[0]

if row["steam_pressure"]<118:

    recommendation.append(
        "Increase Steam Pressure by 2%"
    )

if row["machine_speed"]>980:

    recommendation.append(
        "Reduce Machine Speed by 2%"
    )

if row["stock_flow"]<95:

    recommendation.append(
        "Increase Stock Flow"
    )

if row["moisture"]>6:

    recommendation.append(
        "Reduce Moisture"
    )

recommendation

['Reduce Moisture']

In [16]:
prediction=model.predict(sample)[0]

print("Future Basis Weight")

print(round(prediction,2))

print()

print("Recommendations")

for r in recommendation:

    print("✔",r)

Future Basis Weight
79.63

Recommendations
✔ Reduce Moisture


In [17]:
def recommend(sensor_data):

    prediction=model.predict(sensor_data)[0]

    recommendation=[]

    row=sensor_data.iloc[0]

    if row["steam_pressure"]<118:
        recommendation.append("Increase Steam Pressure by 2%")

    if row["machine_speed"]>980:
        recommendation.append("Reduce Machine Speed by 2%")

    if row["stock_flow"]<95:
        recommendation.append("Increase Stock Flow")

    if row["moisture"]>6:
        recommendation.append("Reduce Moisture")

    return prediction,recommendation

In [18]:
import xgboost
print(xgboost.__version__)

3.3.0


In [22]:
import joblib

joblib.dump(model, "xgboost_model_v2.pkl")

loaded = joblib.load("xgboost_model_v2.pkl")

print(type(loaded))
print("SUCCESS")

<class 'xgboost.sklearn.XGBRegressor'>
SUCCESS


In [19]:
import joblib

loaded_model = joblib.load("xgboost_model_v2.pkl")
print("Model loaded successfully!")

Model loaded successfully!


In [20]:
import joblib

joblib.dump(model, "xgboost_model_v2.pkl", compress=0)

['xgboost_model_v2.pkl']

In [21]:
loaded_model = joblib.load("xgboost_model_v2.pkl")
print("Verified successfully")

Verified successfully


In [23]:
model.save_model("xgboost_model_v2.json")